# Kaggle Synthetic Dataset & ML Experiment Lab

Create **reproducible datasets** for classification, regression, clustering, anomaly detection, and time series. Includes missing values, correlated features, plots, a quick scikit-learn baseline, and CSV/JSON/metadata exports.

**Kaggle steps:** Open Kaggle → **Code** → **New Notebook** → **File > Import Notebook** (or upload the `.ipynb`), then **Run All**. The notebook is self-contained: no internet, secrets, or external files needed. Output is written to `/kaggle/working/` on Kaggle and `./output/` elsewhere. All data is synthetic; never interpret baseline scores as real-world predictive validity.

## 1. Experiment controls
Change these settings and rerun the cells below. `imbalance` is the positive rate for classification and anomaly rate for anomaly detection.

In [ ]:
KIND = 'classification'  # classification | regression | clustering | anomaly | timeseries
ROWS = 1000
FEATURES = 6
SEED = 42
MISSING_RATE = 0.02
NOISE = 0.15
IMBALANCE = 0.30
RUN_BASELINE = True

## 2. Dataset generator and evaluation (standalone code)

In [ ]:
"""Synthetic dataset lab: Kaggle-compatible and runnable as a CLI."""
import argparse
import json
from pathlib import Path

import numpy as np
import pandas as pd

TYPES = ('classification', 'regression', 'clustering', 'anomaly', 'timeseries')


def make_dataset(kind='classification', rows=1000, features=6, seed=42,
                 missing_rate=0.02, noise=0.15, imbalance=0.30):
    """Build repeatable synthetic data for exploration, not real-world validation."""
    if kind not in TYPES:
        raise ValueError(f'kind must be one of {TYPES}')
    if rows < 30 or features < 2:
        raise ValueError('rows >= 30 and features >= 2 required')
    if not 0 <= missing_rate < 1 or not 0 <= noise <= 2 or not 0 < imbalance < 1:
        raise ValueError('Check missing_rate [0,1), noise [0,2], imbalance (0,1)')
    rng = np.random.default_rng(seed)
    X = rng.normal(size=(rows, features))
    X[:, 1] = 0.65 * X[:, 0] + 0.35 * X[:, 1]  # deliberately correlated predictors
    names = [f'feature_{i:02d}' for i in range(1, features + 1)]
    df = pd.DataFrame(X, columns=names)
    weights = rng.normal(size=features)
    signal = X @ weights / np.sqrt(features)
    if kind == 'classification':
        raw = signal + rng.normal(0, noise, rows)
        threshold = np.quantile(raw, 1 - imbalance)
        df['target'] = (raw >= threshold).astype(int)
    elif kind == 'regression':
        df['target'] = 25 + 10 * signal + rng.normal(0, max(noise, 0.001) * 10, rows)
    elif kind == 'clustering':
        centers = rng.normal(0, 3, size=(3, features))
        cluster = rng.integers(0, 3, size=rows)
        X = centers[cluster] + rng.normal(0, 0.65 + noise, size=(rows, features))
        df[names] = X
        df['cluster_truth'] = cluster  # reference only, exclude from training
    elif kind == 'anomaly':
        is_anomaly = rng.random(rows) < imbalance
        X[is_anomaly] += rng.normal(0, 5, size=(int(is_anomaly.sum()), features))
        df[names] = X
        df['is_anomaly'] = is_anomaly.astype(int)  # reference only, exclude from training
    else:
        t = np.arange(rows)
        df.insert(0, 'timestamp', pd.date_range('2025-01-01', periods=rows, freq='h'))
        seasonality = 5 * np.sin(2 * np.pi * t / 24)
        trend = 0.025 * t
        df['target'] = 30 + trend + seasonality + 2 * signal + rng.normal(0, max(noise, .001) * 5, rows)
    # Only predictors contain missing entries; keep labels intact.
    mask = rng.random((rows, features)) < missing_rate
    df.loc[:, names] = df[names].mask(mask)
    return df


def baseline(df, kind='classification', seed=42):
    """Quick demonstration baseline. Synthetic scores do not prove production quality."""
    from sklearn.impute import SimpleImputer
    from sklearn.pipeline import make_pipeline
    from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, IsolationForest
    from sklearn.cluster import KMeans
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import (accuracy_score, mean_absolute_error, adjusted_rand_score,
                                 precision_score, recall_score)
    cols = [c for c in df if c.startswith('feature_')]
    X = df[cols]
    if kind in ('classification', 'regression'):
        stratify = df['target'] if kind == 'classification' else None
        X_train, X_test, y_train, y_test = train_test_split(
            X, df['target'], test_size=0.25, random_state=seed, stratify=stratify)
        model = RandomForestClassifier(n_estimators=75, random_state=seed, n_jobs=-1) if kind == 'classification' else RandomForestRegressor(n_estimators=75, random_state=seed, n_jobs=-1)
        pipe = make_pipeline(SimpleImputer(strategy='median'), model)
        pipe.fit(X_train, y_train)
        pred = pipe.predict(X_test)
        return {'metric': 'accuracy', 'value': float(accuracy_score(y_test, pred))} if kind == 'classification' else {'metric': 'MAE', 'value': float(mean_absolute_error(y_test, pred))}
    if kind == 'clustering':
        XX = SimpleImputer(strategy='median').fit_transform(X)
        pred = KMeans(n_clusters=3, random_state=seed, n_init=10).fit_predict(XX)
        return {'metric': 'adjusted_rand_index', 'value': float(adjusted_rand_score(df['cluster_truth'], pred))}
    if kind == 'anomaly':
        XX = SimpleImputer(strategy='median').fit_transform(X)
        pred = (IsolationForest(contamination=min(max(df['is_anomaly'].mean(), .001), .49), random_state=seed).fit_predict(XX) == -1).astype(int)
        return {'metric': 'anomaly_precision', 'value': float(precision_score(df['is_anomaly'], pred, zero_division=0)), 'recall': float(recall_score(df['is_anomaly'], pred, zero_division=0))}
    # chronological validation, not a random split
    cutoff = int(len(df) * .75)
    pipe = make_pipeline(SimpleImputer(strategy='median'), RandomForestRegressor(n_estimators=75, random_state=seed, n_jobs=-1))
    # Includes trend/time/seasonality features so extrapolation limitations are visible.
    X_time = df[cols].copy()
    t = np.arange(len(df))
    X_time['time_index'] = t
    X_time['hour_sin'] = np.sin(2 * np.pi * t / 24)
    X_time['hour_cos'] = np.cos(2 * np.pi * t / 24)
    pipe.fit(X_time.iloc[:cutoff], df['target'].iloc[:cutoff])
    prediction = pipe.predict(X_time.iloc[cutoff:])
    return {'metric': 'chronological_MAE', 'value': float(mean_absolute_error(df['target'].iloc[cutoff:], prediction))}


def export_dataset(df, kind, seed, directory):
    dest = Path(directory)
    dest.mkdir(parents=True, exist_ok=True)
    csv_path = dest / f'synthetic_{kind}.csv'
    json_path = dest / f'synthetic_{kind}.json'
    metadata_path = dest / f'synthetic_{kind}_metadata.json'
    df.to_csv(csv_path, index=False)
    df.to_json(json_path, orient='records', indent=2, date_format='iso')
    metadata_path.write_text(json.dumps({'kind': kind, 'rows': len(df), 'columns': df.columns.tolist(), 'seed': seed, 'synthetic': True}, indent=2))
    return csv_path, json_path, metadata_path




## 3. Generate and inspect

In [ ]:
df = make_dataset(KIND, ROWS, FEATURES, SEED, MISSING_RATE, NOISE, IMBALANCE)
print('Shape:', df.shape, 'Type:', KIND)
display(df.head(10))
display(df.describe(include='all').T)
print('Missing values per column:\n', df.isna().sum())

## 4. Visualize the dataset

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
cols = [c for c in df.columns if c.startswith('feature_')]
axes[0].hist(df[cols[0]].dropna(), bins=30)
axes[0].set_title(f'Distribution: {cols[0]}')
if KIND in ('classification', 'clustering', 'anomaly'):
    col = {'classification':'target','clustering':'cluster_truth','anomaly':'is_anomaly'}[KIND]
    df[col].value_counts().sort_index().plot(kind='bar', ax=axes[1])
    axes[1].set_title(f'Class counts: {col}')
elif KIND == 'timeseries':
    axes[1].plot(df['timestamp'], df['target'])
    axes[1].tick_params(axis='x', rotation=45)
    axes[1].set_title('Synthetic time series')
else:
    axes[1].scatter(df[cols[0]], df['target'], alpha=.25, s=8)
    axes[1].set_title('Feature vs. target')
plt.tight_layout()
plt.show()
print('Correlation among first features:')
display(df[cols[:min(6,len(cols))]].corr().round(2))

## 5. Train a simple baseline
Labels named `cluster_truth` or `is_anomaly` are used for **evaluation only**, never supplied to the unsupervised model. For time series, the split is chronological.

In [ ]:
if RUN_BASELINE:
    result = baseline(df, KIND, SEED)
    print('Baseline:', result)

## 6. Export CSV, JSON, metadata
On Kaggle, files in `/kaggle/working` appear under **Output** after you run the notebook. To keep notebook outputs, use **Save Version**.

In [ ]:
import os
output_dir = '/kaggle/working' if os.path.isdir('/kaggle/working') else './output'
paths = export_dataset(df, KIND, SEED, output_dir)
print('Files created:')
for path in paths:
    print(path)

## 7. Create a custom dataset from your own formula
Edit the formula/columns for your own domain (latency, GPU utilization, queue depth, tokens/sec, fraud, sensor readings, etc.).

In [ ]:
rng = np.random.default_rng(SEED)
n = 500
custom = pd.DataFrame({
    'request_concurrency': rng.integers(1, 200, size=n),
    'queue_depth': rng.poisson(15, size=n),
    'gpu_utilization_pct': rng.uniform(10, 98, size=n),
})
custom['latency_ms'] = (25 + .7 * custom['request_concurrency'] + 3 * custom['queue_depth']
                        + .12 * custom['gpu_utilization_pct'] + rng.normal(0, 12, size=n)).clip(lower=0)
display(custom.head())
custom.to_csv(Path(output_dir) / 'custom_inference_load.csv', index=False)
print('Saved:', Path(output_dir) / 'custom_inference_load.csv')

## Next experiments
Try comparing RandomForest vs XGBoost/LightGBM, inject concept drift, increase missingness, compare imputation strategies, and track results with MLflow. The custom inference-load dataset is a simulated relationship for testing pipelines—not a physical GPU performance model.